# Anomaly Context — Field Asset Health Monitor Project

Stage 3 of the pipeline. In the program template this stage is "outliers &
missing"; for this dataset both were settled in stages 1–2 (zero missing,
"outliers" proven machine behavior), so the stage's real content is the
**deeper investigation of the documented failures and the open questions**
(OQ1: digital polarity, OQ2: gaps vs failures), producing the group labels
and comparisons that feature engineering (04) will build on.

Consumes: `sensor_readings.parquet`, `failure_windows.csv` (D10 contract).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from fahm import preprocessing as pp
from fahm import plotting as pl

cfg = pp.load_config("../configs/config.yaml")
df = pp.load_processed(cfg)
fw = pd.read_csv(cfg["paths"]["failure_windows"],
                 parse_dates=["start", "end", "maintenance"])
gaps = pp.find_gaps(df, cfg)
print(df.shape, "| failures:", len(fw), "| gaps:", len(gaps))

## 1. Missing values & outliers — status (program-stage requirement)

Settled in earlier stages, cited here:
- **Missing:** zero across all 16 columns (checks table, stage 1).
- **Outliers:** the extremes are machine behavior, not errors — TP2
  zero-offset is calibration; the DV_pressure 9.8 bar sample is a verified
  maintenance episode (stage 2, Q4). Range checks (D08) guard against
  actual sensor breakage.

No cleaning action required → this notebook investigates *anomalies in
context* instead.

## 2. The four failures, seen in time

One multi-sensor timeline per failure: ±2 days around each window, the key
sensors (TP3, Motor_current, DV_eletric, Oil_temperature), window shaded.
Questions per failure: what changes BEFORE the window (early-warning
evidence)? what does DURING look like (continuous load? faster cycling)?
what happens AFTER (repair signature, gap)?

In [ ]:
# TODO: helper in plotting.py — plot_failure_context(df, fw_row, sensors, pad_days=2)
#       (generalizes plot_sensor_timeline: several sensors stacked, axvspan for
#        the window, second span for the maintenance date if present)
# for _, f in fw.iterrows():
#     fig = pl.plot_failure_context(df, f, ["TP3","Motor_current","DV_eletric","Oil_temperature"])
#     display(fig)
#     pl.save_fig(fig, f"failure_context_{f.failure_id}", cfg)

### Per-failure observations
- **F1:** <what the timeline shows>
- **F2:** <...>
- **F3:** <...>
- **F4:** <...>

## 3. OQ2 — gaps vs failures

Exhibit A (found in stage 1's inventory): a 1,288-min gap starts
**2020-06-07 14:19** — 11 minutes before F3's documented end (14:30) — and
recording resumes the morning of F3's maintenance day. Recording stops
around repairs.

Systematic test: for every failure, list gaps that start/end within ± N
days of the window; and separately, does every LARGE gap sit near a
failure/maintenance date, or do unexplained ones remain (candidates for
undocumented events, like the Apr 6 episode)?

In [ ]:
# TODO: small analysis (notebook-level is fine, promote to src if reused):
# for each fw row: gaps[(gaps.gap_start > f.start - pd.Timedelta(days=3)) &
#                        (gaps.gap_end   < f.end   + pd.Timedelta(days=3))]
# then the reverse: top-10 gaps -> nearest failure/maintenance distance

### OQ2 verdict
<gaps cluster around failures: yes/no/partially; unexplained large gaps: ...;
consequence for features (post-gap warm-up rule?): ...>

## 4. OQ1 — digital polarity (Oil_level, Pressure_switch)

Both are "active" ~90-99% of the time despite docs saying active = fault /
event. Test: WHEN does the inactive time occur?

In [ ]:
# inactive_ts = df.loc[df["Oil_level"] == 0, "timestamp"]
# 1) daily counts over the whole record (bar/line) with failure windows shaded
# 2) fraction of inactive time inside failure windows ± 1 day vs outside
# same for Pressure_switch

### OQ1 verdict
<clustered around failures/maintenance/gaps -> polarity/behavior story;
scattered -> revisit. Sensor-table meaning updated accordingly.>

## 5. Healthy vs pre-failure groups (the t-test, adapted)

Program recipe: t-test features across target classes. Adapted here (D13):
rows are labeled from `fw` (healthy / pre-failure / in-failure /
post-repair), groups compared by **effect size and distribution overlap**,
not p-values — 10s samples are heavily autocorrelated, so classic
significance is meaningless at n=1.5M; rank-based stats (Mann-Whitney) may
be reported as descriptive only. The REAL significance test is the model's
early-warning evaluation later.

In [ ]:
# TODO: labeling helper in src (this one WILL be reused by stage 4):
# def label_windows(df, fw, prefail_hours=48, postrepair_hours=12) -> pd.Series
#   categories: healthy / prefail / infail / postrepair
# labels = pp.label_windows(df, fw)
# labels.value_counts()

In [ ]:
# comparison per key sensor (Oil_temperature, Motor_current, TP3, DV_eletric):
# side-by-side hists or .groupby(labels).describe() ; effect size =
# (median_prefail - median_healthy) / IQR_healthy  (robust, state-mixture friendly)

### Group-comparison findings
<which sensors shift before failure, by how much — this is the
early-warning-detectability evidence stage 4 builds features from>

## Findings feed-forward
- **To 04 (features):** label_windows() helper; sensors with pre-failure
  shift: <...>; post-gap warm-up rule: <needed/not, from OQ2>.
- **Sensor table updates:** Oil_level / Pressure_switch meaning: <from OQ1>.
- **Open items:** <anything unresolved>